In [1]:
"""
Standalone calculation of effective radius (r_eff) from volumetric ice radius (r_ice_vol),
exactly replicating the parameterization used in CoCiP (pycontrails).

The habit-weighted r_eff calculation mirrors:
  pycontrails/models/cocip/radiative_forcing.py

References
----------
- Schumann et al. (2011): "Effective radius of ice particles fetched from satellite"
  https://doi.org/10.5194/acp-11-18571-2011
- Schumann et al. (2012): "Parametric radiative forcing of contrail cirrus"
"""

import numpy as np
import numpy.typing as npt


# ---------------------------------------------------------------------------
# Default habit distributions and radius thresholds
# (CocipParams defaults from pycontrails)
# ---------------------------------------------------------------------------

# fmt: off
DEFAULT_HABIT_DISTRIBUTIONS = np.array([
    # Sphere  SolCol  HolCol  RghAgg  Rose-6  Plate  Droxtal  Myhre
    [0.0,    0.0,    0.0,    0.0,    0.0,    0.0,   1.0,     0.0],   # r < 5 µm
    [0.0,    0.3,    0.0,    0.0,    0.0,    0.0,   0.7,     0.0],   # 5 ≤ r < 9.5
    [0.0,    0.3,    0.3,    0.0,    0.0,    0.0,   0.4,     0.0],   # 9.5 ≤ r < 18
    [0.0,    0.0,    0.0,    0.5,    0.3,    0.0,   0.2,     0.0],   # 18 ≤ r < 30
    [0.0,    0.0,    0.0,    0.45,   0.45,   0.1,   0.0,     0.0],   # 30 ≤ r < 50
    [0.0,    0.0,    0.0,    0.15,   0.3,    0.55,  0.0,     0.0],   # r ≥ 50
])
# fmt: on

DEFAULT_RADIUS_THRESHOLD_UM = np.array([5.0, 9.5, 18.0, 30.0, 50.0])


# ---------------------------------------------------------------------------
# Habit index lookup
# ---------------------------------------------------------------------------

def habit_weight_regime_idx(
    r_vol_um: npt.NDArray[np.floating],
    radius_threshold_um: npt.NDArray[np.floating],
) -> npt.NDArray[np.intp]:
    """Return the row index into the habit_distributions array for each particle radius."""
    idx = np.digitize(r_vol_um, radius_threshold_um)
    idx[np.isnan(r_vol_um)] = 0
    return idx


def habit_weights(
    r_vol_um: npt.NDArray[np.floating],
    habit_distributions: npt.NDArray[np.floating] = DEFAULT_HABIT_DISTRIBUTIONS,
    radius_threshold_um: npt.NDArray[np.floating] = DEFAULT_RADIUS_THRESHOLD_UM,
) -> npt.NDArray[np.floating]:
    """Assign weights to different ice particle habits for each radius value."""
    if not np.allclose(np.sum(habit_distributions, axis=1), 1.0, atol=1e-3):
        raise ValueError("Habit weight distributions must sum to 1 across columns")
    if habit_distributions.shape[0] != (radius_threshold_um.size + 1):
        raise ValueError(
            "The number of rows in `habit_distributions` must equal 1 + "
            "the size of `radius_threshold_um`"
        )
    idx = habit_weight_regime_idx(r_vol_um, radius_threshold_um)
    return habit_distributions[idx]


# ---------------------------------------------------------------------------
# Per-habit effective radius parameterisations
# (Table 3 of Schumann et al. 2011)
# ---------------------------------------------------------------------------

def effective_radius_sphere(r_vol_um: npt.NDArray[np.floating]) -> npt.NDArray[np.floating]:
    """Habit 0 — Sphere."""
    return np.minimum(r_vol_um, 25.0)


def effective_radius_solid_column(r_vol_um: npt.NDArray[np.floating]) -> npt.NDArray[np.floating]:
    """Habit 1 — Solid column."""
    r_eff_um = (
        0.2588 * np.exp(-(6.912e-3 * r_vol_um))
        + 0.6372 * np.exp(-(3.142e-4 * r_vol_um))
    ) * r_vol_um
    is_small = r_vol_um <= 42.2
    r_eff_um[is_small] = 0.824 * r_vol_um[is_small]
    return np.minimum(r_eff_um, 45.0)


def effective_radius_hollow_column(r_vol_um: npt.NDArray[np.floating]) -> npt.NDArray[np.floating]:
    """Habit 2 — Hollow column."""
    r_eff_um = (
        0.2281 * np.exp(-(7.359e-3 * r_vol_um))
        + 0.5651 * np.exp(-(3.350e-4 * r_vol_um))
    ) * r_vol_um
    is_small = r_vol_um <= 39.7
    r_eff_um[is_small] = 0.729 * r_vol_um[is_small]
    return np.minimum(r_eff_um, 45.0)


def effective_radius_rough_aggregate(r_vol_um: npt.NDArray[np.floating]) -> npt.NDArray[np.floating]:
    """Habit 3 — Rough aggregate."""
    return np.minimum(0.574 * r_vol_um, 45.0)


def effective_radius_rosette(r_vol_um: npt.NDArray[np.floating]) -> npt.NDArray[np.floating]:
    """Habit 4 — Rosette-6."""
    r_eff_um = r_vol_um * (
        0.1770 * np.exp(-(2.144e-2 * r_vol_um))
        + 0.4267 * np.exp(-(3.562e-4 * r_vol_um))
    )
    return np.minimum(r_eff_um, 45.0)


def effective_radius_plate(r_vol_um: npt.NDArray[np.floating]) -> npt.NDArray[np.floating]:
    """Habit 5 — Plate."""
    r_eff_um = r_vol_um * (
        0.1663
        + 0.3713 * np.exp(-(0.0336 * r_vol_um))
        + 0.3309 * np.exp(-(0.0035 * r_vol_um))
    )
    return np.minimum(r_eff_um, 45.0)


def effective_radius_droxtal(r_vol_um: npt.NDArray[np.floating]) -> npt.NDArray[np.floating]:
    """Habit 6 — Droxtal."""
    return np.minimum(0.94 * r_vol_um, 45.0)


def effective_radius_myhre(r_vol_um: npt.NDArray[np.floating]) -> npt.NDArray[np.floating]:
    """Habit 7 — Myhre."""
    return np.minimum(r_vol_um, 45.0)


def effective_radius_by_habit(
    r_vol_um: npt.NDArray[np.floating],
    habit_idx: npt.NDArray[np.intp],
) -> npt.NDArray[np.floating]:
    """
    Calculate r_eff for a flat array of (r_vol_um, habit_idx) pairs.

    This is the vectorised piecewise function used internally in CoCiP's
    longwave_radiative_forcing and shortwave_radiative_forcing.
    """
    cond_list = [
        habit_idx == 0,
        habit_idx == 1,
        habit_idx == 2,
        habit_idx == 3,
        habit_idx == 4,
        habit_idx == 5,
        habit_idx == 6,
        habit_idx == 7,
    ]
    func_list = [
        effective_radius_sphere,
        effective_radius_solid_column,
        effective_radius_hollow_column,
        effective_radius_rough_aggregate,
        effective_radius_rosette,
        effective_radius_plate,
        effective_radius_droxtal,
        effective_radius_myhre,
        0.0,  # fallback (should never be reached)
    ]
    return np.piecewise(r_vol_um, cond_list, func_list)  # type: ignore[call-overload]


# ---------------------------------------------------------------------------
# Main public function
# ---------------------------------------------------------------------------

def r_eff_from_r_ice_vol(
    r_ice_vol: npt.ArrayLike,
    habit_distributions: npt.NDArray[np.floating] = DEFAULT_HABIT_DISTRIBUTIONS,
    radius_threshold_um: npt.NDArray[np.floating] = DEFAULT_RADIUS_THRESHOLD_UM,
) -> npt.NDArray[np.floating]:
    """
    Calculate the habit-weighted effective radius (r_eff) from the volumetric
    ice radius (r_ice_vol), exactly as CoCiP does it.

    Parameters
    ----------
    r_ice_vol : array-like
        Contrail ice particle volume mean radius [m].
        CoCiP stores this in metres; it is converted to µm internally.
    habit_distributions : ndarray, optional
        Weights for each of the 8 ice habits across radius bins.
        Defaults to CocipParams().habit_distributions.
    radius_threshold_um : ndarray, optional
        Radius bin boundaries [µm].
        Defaults to CocipParams().radius_threshold_um.

    Returns
    -------
    r_eff : ndarray
        Habit-weighted effective radius [m], same shape as `r_ice_vol`.

    Notes
    -----
    Inside CoCiP the weighted r_eff is computed per habit in
    `effective_radius_by_habit`, then each habit's RF contribution is
    weighted by `habit_weights_` and summed.  Here we reproduce that
    weighted average of r_eff directly so you can inspect it independently.
    """
    r_ice_vol = np.asarray(r_ice_vol, dtype=float)
    scalar_input = r_ice_vol.ndim == 0
    r_ice_vol = np.atleast_1d(r_ice_vol)

    # Convert m → µm (CoCiP works in µm inside the RF functions)
    r_vol_um = r_ice_vol * 1e6

    # Habit weights: shape (n_waypoints, 8)
    weights = habit_weights(r_vol_um, habit_distributions, radius_threshold_um)

    # Compute r_eff for every (waypoint, habit) combination where weight > 0
    habit_weight_mask = weights > 0.0
    idx0, idx1 = np.nonzero(habit_weight_mask)

    r_vol_um_h = r_vol_um[idx0]
    r_eff_um_h = effective_radius_by_habit(r_vol_um_h, idx1)

    # Weighted sum over habits → one r_eff per waypoint
    r_eff_um_weighted = np.zeros_like(weights)
    r_eff_um_weighted[idx0, idx1] = r_eff_um_h * weights[habit_weight_mask]
    r_eff_um = np.sum(r_eff_um_weighted, axis=1)

    # Convert µm → m to match CoCiP's output convention
    r_eff = r_eff_um * 1e-6

    return r_eff[0] if scalar_input else r_eff


# ---------------------------------------------------------------------------
# Quick demonstration
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # Example: a range of volumetric radii from 1 µm to 80 µm (in meters)
    r_ice_vol_m = np.array([1, 3, 5, 9.5, 18, 30, 50, 80]) * 1e-6

    r_eff_m = r_eff_from_r_ice_vol(r_ice_vol_m)

    print(f"{'r_ice_vol (µm)':>16}  {'r_eff (µm)':>12}")
    print("-" * 32)
    for rv, re in zip(r_ice_vol_m * 1e6, r_eff_m * 1e6):
        print(f"{rv:>16.2f}  {re:>12.4f}")

  r_ice_vol (µm)    r_eff (µm)
--------------------------------
            1.00        0.9400
            3.00        2.8200
            5.00        4.5260
            9.50        8.5994
           18.00       11.4892
           30.00       18.8868
           50.00       26.2760
           80.00       36.8997
